In [ ]:
# 1. Install YOLOv11 core library
!pip install -q ultralytics

In [ ]:
import os
import shutil
from ultralytics import YOLO

# 1. Initialize the YOLO11-Small model (Better feature extraction for tiny license plates)
print("[*] Initializing YOLO11-Small (Professional Grade)...")
model = YOLO("yolo11s.pt")

# 2. Configure EXACT dataset path
base_path = "/kaggle/input/datasets/tranquangduyenk17ct/vietnam-license-plate-v1"
yaml_path = os.path.join(base_path, "data.yaml")

# 3. Start the Advanced Fine-tuning process
if os.path.exists(yaml_path):
    print(f"[*] Found data.yaml at: {yaml_path}")
    print("[*] Starting the Academic Training Pipeline...")
    
    results = model.train(
        data=yaml_path,
        epochs=150,                 # Deep learning cycles
        patience=30,                # Early stopping mechanism to prevent overfitting
        imgsz=640,                  # Optimal resolution for balance
        batch=32,                   # Maximize GPU utilization
        project="/kaggle/working/SentinelAI",
        name="plate_model_pro",
        
        # --- ACADEMIC DATA AUGMENTATION FOR LICENSE PLATES ---
        mosaic=1.0,                 # Combines 4 images (Crucial for learning context around plates)
        degrees=15.0,               # Random rotation (+/- 15 deg) for tilted plates
        perspective=0.001,          # [CRITICAL] Simulates 3D angled CCTV viewpoints
        shear=2.0,                  # Slants the image to mimic moving vehicles
        scale=0.6,                  # Random zoom to force detection of tiny plates
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, # Lighting variations (Night/Day/Glare)
        fliplr=0.0,                 # [CRITICAL] DISABLED! License plates cannot be horizontally flipped
        
        optimizer="auto",
        cos_lr=True                 # Cosine learning rate decay for smooth convergence
    )
    print("[+] ADVANCED TRAINING COMPLETE! Optimal weights are safely saved.")
else:
    print(f"[!] FATAL ERROR: 'data.yaml' not found at {yaml_path}")

In [ ]:
import glob

# 1. Auto-locate the newly trained optimal weights
weight_files = glob.glob("/kaggle/working/SentinelAI/plate_model_pro/weights/best.pt")
best_weight = weight_files[0] if weight_files else None

if not best_weight:
    print("[!] ERROR: Trained weights not found. Did the training finish successfully?")
else:
    print(f"[*] Loading the Ultimate Plate Model for Evaluation: {best_weight}")
    eval_model = YOLO(best_weight)
    
    # 2. Execute strict validation on unseen 'test' data
    print("[*] Running Strict Evaluation on Test Dataset...")
    metrics = eval_model.val(
        data=yaml_path, 
        split='test', 
        project="/kaggle/working/SentinelAI", 
        name="plate_eval_results"
    )
    
    # 3. Display Professional Report
    print("\n" + "="*60)
    print("LICENSE PLATE DETECTION - EVALUATION REPORT")
    print("="*60)
    print(f"mAP@50 (Standard Detection Accuracy) : {metrics.box.map50:.4f}")
    print(f"mAP@50-95 (Strict Bounding Box fit)  : {metrics.box.map:.4f}")
    print(f"Precision (Accuracy when it detects) : {metrics.box.mp:.4f}")
    print(f"Recall (Ability to find all plates)  : {metrics.box.mr:.4f}")
    print("="*60)

In [ ]:
from IPython.display import Image, display, HTML
import os

# Define export directories
train_dir = "/kaggle/working/SentinelAI/plate_model_pro"
eval_dir = "/kaggle/working/SentinelAI/plate_eval_results"

# Key charts for Academic Review
charts = [
    {"file": "results.png", "path": train_dir, "title": "1. Loss & Convergence Curves", "desc": "Monitoring the learning curve. Ensures validation loss converges properly without overfitting."},
    {"file": "confusion_matrix.png", "path": eval_dir, "title": "2. Confusion Matrix", "desc": "Evaluates False Positives vs False Negatives specifically for license plates."},
    {"file": "PR_curve.png", "path": eval_dir, "title": "3. Precision-Recall Curve", "desc": "Demonstrates the trade-off. An AUC (Area Under Curve) close to 1.0 represents a flawless model."},
    {"file": "F1_curve.png", "path": eval_dir, "title": "4. F1-Score Curve", "desc": "Helps in selecting the optimal confidence threshold for real-world deployment."}
]

print("VISUAL ANALYTICS & ACADEMIC DIAGNOSTICS")
print("-" * 60)

for chart in charts:
    chart_path = os.path.join(chart["path"], chart["file"])
    
    if not os.path.exists(chart_path) and chart["path"] == eval_dir:
        chart_path = os.path.join(train_dir, chart["file"])
        
    if os.path.exists(chart_path):
        display(HTML(f"<h3>{chart['title']}</h3><p><i>{chart['desc']}</i></p>"))
        display(Image(filename=chart_path, width=900))
        print("\n")
    else:
        print(f"[!] Pending: {chart['file']} (Will be available after evaluation completes)")

In [ ]:
import random
from IPython.display import Image, display

print("[*] Running Random Inference Test on Unseen Data...")

if not best_weight:
    print("[!] ERROR: No trained weights available for testing.")
else:
    # 1. Locate test images directory
    test_images_dir = os.path.join(base_path, "test", "images")
    
    if os.path.exists(test_images_dir):
        # 2. Get all valid image files
        test_images = [f for f in glob.glob(os.path.join(test_images_dir, "*")) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        if test_images:
            # 3. Pick a random image to prevent bias
            sample_image = random.choice(test_images)
            image_name = os.path.basename(sample_image)
            print(f"[*] Selected Sample Image: {image_name}")
            
            # 4. Load the optimal model and execute prediction
            test_model = YOLO(best_weight)
            results = test_model.predict(
                source=sample_image,
                conf=0.25,                  # Test threshold for license plates
                save=True,                  # Save annotated image
                project="/kaggle/working/SentinelAI",
                name="plate_test_inference",
                exist_ok=True
            )
            
            # 5. Render the annotated output image in the notebook
            result_img_path = os.path.join("/kaggle/working/SentinelAI/plate_test_inference", image_name)
            print("\n[+] INFERENCE RESULT:")
            display(Image(filename=result_img_path, width=800))
            
        else:
            print(f"[!] WARNING: No image files found in {test_images_dir}")
    else:
        print(f"[!] ERROR: Test directory not found at {test_images_dir}")

In [ ]:
import shutil
from IPython.display import FileLink

print("[*] Archiving the newly trained License Plate weights...")
output_dir = "/kaggle/working/SentinelAI/plate_model_pro/weights"
zip_path = "/kaggle/working/plate_detector_weights_pro" 

if os.path.exists(output_dir):
    shutil.make_archive(zip_path, 'zip', output_dir)
    print(f"[+] Archive created successfully at: {zip_path}.zip")
    print("[!] ACTION: Click the link below to download your PRO weights for Local Deployment.")
    display(FileLink(r'plate_detector_weights_pro.zip'))
else:
    print("[!] ERROR: Weight directory not found. Please ensure training completed successfully.")